In [1]:
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq

from dotenv import load_dotenv

In [2]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [3]:
kb_path = Path("../data/knowledge_base")

print("Knowledge base:", kb_path)
print()

for file in kb_path.glob("*.md"):
    print(file.name)

Knowledge base: ../data/knowledge_base

attendance_policy.md
working_hours_policy.md
overtime_policy.md
sick_leave_policy.md
leave_policy.md
remote_work_policy.md


In [4]:
documents = []

for file in kb_path.glob("*.md"):

    text = file.read_text(encoding="utf-8")

    document = Document(
        page_content=text,
        metadata={
            "source": file.name
        }
    )

    documents.append(document)

print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [5]:
for document in documents:

    print("=" * 60)
    print("SOURCE:", document.metadata["source"])
    print("=" * 60)

    print(document.page_content[:500])
    print()

SOURCE: attendance_policy.md
# NexaTech GmbH Attendance Policy

## Purpose
This policy defines how employees record working time and office attendance.

## Office Attendance Requirement
Employees assigned to hybrid work are expected to work from the office at least three days per week unless an approved exception applies.

## Attendance Recording
Employees must record their check-in and check-out times using the company attendance system.

## Missing Records
If an employee forgets to record a check-in or check-out, they sho

SOURCE: working_hours_policy.md
# NexaTech GmbH Working Hours Policy

## Standard Working Week
Full-time employees normally work 40 hours per week. Part-time employees follow the weekly hours stated in their employment contract.

## Daily Working Hours
A standard full-time working day is approximately 8 hours excluding breaks.

## Core Hours
Core working hours are from 09:00 to 15:00.

## Flexible Working
Employees may normally start between 07:00 and 09:00 and fin

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ".",
        " "
    ]
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")


Created 8 chunks.


In [7]:
for i, chunk in enumerate(chunks[:5]):

    print("=" * 60)
    print(f"CHUNK {i + 1}")
    print("SOURCE:", chunk.metadata["source"])
    print("=" * 60)

    print(chunk.page_content)
    print()

CHUNK 1
SOURCE: attendance_policy.md
# NexaTech GmbH Attendance Policy

## Purpose
This policy defines how employees record working time and office attendance.

## Office Attendance Requirement
Employees assigned to hybrid work are expected to work from the office at least three days per week unless an approved exception applies.

## Attendance Recording
Employees must record their check-in and check-out times using the company attendance system.

## Missing Records
If an employee forgets to record a check-in or check-out, they should submit an attendance correction request to their manager.

## Late Arrival
Employees arriving after 09:00 should notify their manager when required by their team.

CHUNK 2
SOURCE: attendance_policy.md
## Late Arrival
Employees arriving after 09:00 should notify their manager when required by their team.

## Corrections
Attendance corrections should normally be submitted within five working days.

## Business Travel
Approved business travel is considered a

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

/Users/aaravmenon/New folder/Attendance RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5591.75it/s]


Embedding model loaded.


In [10]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nexatech_policies"
)

print(
    f"Vector store ready. "
    f"{vector_store._collection.count()} vectors stored."
)

Vector store ready. 8 vectors stored.


In [12]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 4
    }
)

print("Retriever ready.")

Retriever ready.


In [13]:
query = "How many days can employees work from home?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):

    print("=" * 60)
    print(f"RESULT {i}")
    print("SOURCE:", doc.metadata["source"])
    print("=" * 60)

    print(doc.page_content)
    print()

RESULT 1
SOURCE: remote_work_policy.md
# NexaTech GmbH Remote Work Policy

## Eligibility
Full-time employees may work remotely if their role permits remote work.

## Weekly Remote Work Limit
Employees may normally work remotely for a maximum of two regular working days per week.

## Office Requirement
Employees assigned to hybrid work are expected to work from the office for at least three days per week.

## Core Hours
Employees working remotely must remain available between 09:00 and 15:00.

## Exceptions
Managers may approve temporary exceptions for business, family, or other justified circumstances.

## Attendance
Remote work days count as working days but do not count toward the minimum office attendance requirement.

RESULT 2
SOURCE: working_hours_policy.md
# NexaTech GmbH Working Hours Policy

## Standard Working Week
Full-time employees normally work 40 hours per week. Part-time employees follow the weekly hours stated in their employment contract.

## Daily Working Hours
A sta

In [14]:
test_questions = [
    "How many days can employees work from home?",
    "What should I do if I forget to check out?",
    "How many hours does a full-time employee normally work?",
    "How does overtime work?",
    "What should I do if I am sick?",
    "How many vacation days do full-time employees receive?"
]

for question in test_questions:

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("=" * 70)

    retrieved_docs = retriever.invoke(question)

    for i, doc in enumerate(retrieved_docs, 1):

        print(
            f"\nResult {i} "
            f"[{doc.metadata['source']}]"
        )

        print(doc.page_content[:300])


QUESTION: How many days can employees work from home?

Result 1 [remote_work_policy.md]
# NexaTech GmbH Remote Work Policy

## Eligibility
Full-time employees may work remotely if their role permits remote work.

## Weekly Remote Work Limit
Employees may normally work remotely for a maximum of two regular working days per week.

## Office Requirement
Employees assigned to hybrid work a

Result 2 [working_hours_policy.md]
# NexaTech GmbH Working Hours Policy

## Standard Working Week
Full-time employees normally work 40 hours per week. Part-time employees follow the weekly hours stated in their employment contract.

## Daily Working Hours
A standard full-time working day is approximately 8 hours excluding breaks.

##

Result 3 [leave_policy.md]
# NexaTech GmbH Leave Policy

## Vacation
Full-time employees receive between 28 and 30 paid vacation days per calendar year depending on their employment contract. Part-time employees receive a pro-rated entitlement according to their contract.

In [15]:
llm = ChatGroq(

    model="openai/gpt-oss-120b",

    temperature=0,

    reasoning_format="parsed"

)



print("LLM loaded.")



LLM loaded.


In [16]:
def format_docs(docs):

    formatted_documents = []

    for doc in docs:

        formatted_documents.append(
            f"Source: {doc.metadata['source']}\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted_documents)

In [17]:
SYSTEM_PROMPT = """
You are the NexaTech Workplace Policy Assistant.

Your job is to answer questions about NexaTech GmbH
using the provided company policy documents.

Rules:

1. Answer using ONLY the provided context.
2. Do not invent company policies.
3. If the answer cannot be found in the context,
   say that the available company policies do not
   contain enough information.
4. Keep the answer clear and concise.
5. At the end, mention the source document(s) used.

Context:

{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

print("Prompt created.")

Prompt created.


In [18]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [19]:
question = "How many days can I work from home per week?"

answer = rag_chain.invoke(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
How many days can I work from home per week?

ANSWER:
You may normally work remotely for up to **two regular working days per week**. This limit is set in the Remote Work Policy, which also requires hybrid employees to be in the office at least three days per week.

**Source:** remote_work_policy.md – “Weekly Remote Work Limit: Employees may normally work remotely for a maximum of two regular working days per week.”


In [20]:
questions = [
    "How many days can I work from home per week?",
    "What should I do if I forget to check out?",
    "How many hours does a full-time employee normally work?",
    "What is the company's overtime policy?",
    "What should I do if I am sick?",
    "How many vacation days do full-time employees receive?"
]

for question in questions:

    print("\n" + "=" * 70)
    print("QUESTION:", question)

    answer = rag_chain.invoke(question)

    print("\nANSWER:")
    print(answer)


QUESTION: How many days can I work from home per week?

ANSWER:
You may normally work remotely for up to **two regular working days per week**. This limit is set in the Remote Work Policy, which also requires hybrid employees to be in the office at least three days per week.

**Source:** remote_work_policy.md – “Weekly Remote Work Limit: Employees may normally work remotely for a maximum of two regular working days per week.”

QUESTION: What should I do if I forget to check out?

ANSWER:
If you forget to record a check‑out, you should submit an attendance‑correction request to your manager (and do so within five working days).  

**Source(s):** attendance_policy.md – “Missing Records” and “Corrections”.

QUESTION: How many hours does a full-time employee normally work?

ANSWER:
A full‑time employee at NexaTech normally works **40 hours per week** (approximately 8 hours per day, excluding breaks).

**Source:** working_hours_policy.md.

QUESTION: What is the company's overtime policy?



In [22]:
question = "What is the salary of a Software Engineer at NexaTech?"

answer = rag_chain.invoke(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

QUESTION:
What is the salary of a Software Engineer at NexaTech?

ANSWER:
The available company policies do not contain information about salary levels for a Software Engineer at NexaTech.

**Source(s) consulted:** working_hours_policy.md, overtime_policy.md, remote_work_policy.md, leave_policy.md.
